In [1]:
# matching logic, building on what the exploration found.
# three steps: validate imo -> merge valid imos -> fuzzy match the keyless rows.
# resolution is procedural so it's python; search at the end is sql.
import pandas as pd
import json
from difflib import SequenceMatcher

# same two-draught rename as notebook 1
df = pd.read_csv("../data/case_study_dataset_202509152039.csv").rename(
    columns={"draught": "draught_static", "draught.1": "draught_ais"})
df.shape

(1734, 50)

In [2]:
# step 1: validate imo (7 digits + check digit). also split invalid into
# sentinel (placeholder reused across rows) vs dirty (one-off typo) - they're
# handled differently: no-key vs recoverable.

def imo_is_valid(imo):
    s = str(int(imo))
    if len(s) != 7:
        return False
    # check digit: sum of first 6 digits weighted 7,6,5,4,3,2, mod 10
    total = 0
    for i in range(6):
        total += int(s[i]) * (7 - i)
    check_digit = total % 10
    return check_digit == int(s[6])

df["imo_valid"] = df["imo"].apply(imo_is_valid)

# find the placeholder values: an invalid imo that appears on more than one row
invalid_imos = df.loc[~df["imo_valid"], "imo"]
counts = invalid_imos.value_counts()
sentinels = set()
for imo_value, n in counts.items():
    if n > 1:
        sentinels.add(imo_value)

def classify(row):
    if row["imo_valid"]:
        return "valid"
    elif row["imo"] in sentinels:
        return "sentinel"
    else:
        return "dirty"

df["imo_class"] = df.apply(classify, axis=1)
df["imo_class"].value_counts()
# valid -> merge by imo (step 2); sentinel/dirty -> fuzzy match (step 3)

imo_class
valid       1064
sentinel     617
dirty         53
Name: count, dtype: int64

In [3]:
# step 2: same valid imo = same ship (dups are only mmsi changes, per notebook 1).
# collapse a ship's rows into one record.
#   immutable fields (length, tonnage) shouldn't change -> if they disagree it's an
#     error; take the most common value and flag the disagreement.
#   mutable fields (name, flag, mmsi) do change -> keep the latest as current and
#     store the past values as history.
IMMUTABLE = ["length", "width", "grossTonnage", "netTonnage",
             "builtYear", "depth", "draught_static"]
MUTABLE = ["name", "flag", "mmsi", "callsign"]

def golden_record(group):
    group = group.sort_values("staticData_updateTimestamp")   # oldest -> newest
    record = {"imo": int(group["imo"].iloc[0]), "n_source_rows": len(group)}

    conflicts = {}
    for col in IMMUTABLE:
        values = group[col].dropna()
        if len(values) == 0:
            record[col] = None
            continue
        most_common = values.mode()          # most frequent value(s)
        record[col] = most_common.iloc[-1]   # if tied, take the later one
        if values.nunique() > 1:             # immutable field but values disagree
            conflicts[col] = sorted(values.unique().tolist())
    record["conflicts"] = conflicts

    history = {}
    for col in MUTABLE:
        values = group[col].dropna()
        if len(values) == 0:
            record[col] = None
            continue
        record[col] = values.iloc[-1]        # latest value = current
        distinct = []                        # past values, de-duplicated, in order
        for v in values.tolist():
            if v not in distinct:
                distinct.append(v)
        if len(distinct) > 1:
            history[col] = distinct
    record["history"] = history
    return record

# LADY DUVERA: length disagrees (immutable error), flag/mmsi changed (kept as history)
demo = golden_record(df[(df["imo"] == 1006568) & df["imo_valid"]])
for k in ["imo", "n_source_rows", "name", "flag", "mmsi", "length", "width"]:
    print(k, ":", demo[k])
print("conflicts:", demo["conflicts"])
print("history  :", demo["history"])

imo : 1006568
n_source_rows : 3
name : LADY DUVERA
flag : NL
mmsi : 244076672
length : 47.0
width : 9.0
conflicts: {'length': [44.0, 47.0], 'width': [8.0, 9.0]}
history  : {'flag': ['ES', 'NL'], 'mmsi': [224650177, 244650177, 244076672], 'callsign': ['PB2875', 'PD2511']}


In [4]:
# run golden_record on every valid imo -> the resolved vessel table
valid_df = df[df["imo_valid"]]

records = []
for imo_value, group in valid_df.groupby("imo"):
    records.append(golden_record(group))
gold = pd.DataFrame(records)

merged_count = (gold["n_source_rows"] > 1).sum()
with_mmsi_history = gold["history"].apply(lambda h: "mmsi" in h).sum()
with_conflict = (gold["conflicts"].str.len() > 0).sum()
print(len(valid_df), "valid rows ->", len(gold), "vessels")
print("  merged from >1 row     :", merged_count)
print("  with mmsi history      :", with_mmsi_history)
print("  with immutable conflict:", with_conflict)

# a few that were actually merged
gold[gold["n_source_rows"] > 1].head(5)[
    ["imo", "n_source_rows", "name", "flag", "mmsi", "length", "conflicts"]]

1064 valid rows -> 668 vessels
  merged from >1 row     : 196
  with mmsi history      : 196
  with immutable conflict: 78


,imo,n_source_rows,name,flag,mmsi,length,conflicts
0,1006568,3,LADY DUVERA,NL,244076672,47.0,"{'length': [44.0, 47.0], 'width': [8.0, 9.0]}"
5,1014008,2,JAL KALP,SG,563269500,183.0,{}
7,1015105,3,SETARE SAHEL,KM,620999634,60.0,{}
10,1030727,3,LIDIA,RU,273254690,142.0,"{'grossTonnage': [6652.0, 6732.0], 'netTonnage..."
11,1033004,2,PGMC 7,BZ,312841000,86.0,"{'length': [31.0, 86.0], 'width': [8.0, 17.0]}"


In [5]:
# step 3: keyless rows (sentinel/dirty) have no key but usually a name + dims.
# score each against the golden records on static fields only (never AIS position).
# above the threshold -> link to that vessel; below -> treat as a new vessel
# (don't force a merge: a wrong merge is worse than a missed one).

def clean(s):
    if pd.isna(s):
        return ""
    return str(s).upper().strip()

def name_similarity(a, b):
    a, b = clean(a), clean(b)
    if a == "" or b == "":
        return 0.0
    return SequenceMatcher(None, a, b).ratio()   # 0..1, simple string overlap

def numbers_close(a, b, tol=0.05):
    if pd.isna(a) or pd.isna(b):
        return False                             # no evidence
    bigger = max(abs(a), abs(b))
    if bigger == 0:
        return True
    return abs(a - b) / bigger <= tol            # within 5%

WEIGHTS = {"name": 0.45, "length": 0.20, "width": 0.15,
           "grossTonnage": 0.10, "callsign": 0.10}

def match_score(row, candidate):
    score = 0.0
    reasons = []
    sim = name_similarity(row["name"], candidate["name"])
    if sim > 0:
        score += WEIGHTS["name"] * sim
        reasons.append("name~%.2f" % sim)
    for col in ["length", "width", "grossTonnage"]:
        if numbers_close(row[col], candidate[col]):
            score += WEIGHTS[col]
            reasons.append(col + "=")
    if clean(row["callsign"]) != "" and clean(row["callsign"]) == clean(candidate["callsign"]):
        score += WEIGHTS["callsign"]
        reasons.append("callsign=")
    return round(score, 3), reasons

def best_match(row, threshold=0.6):
    # blocking: only compare against vessels of the same type
    same_type_imos = valid_df[valid_df["vessel_type"] == row["vessel_type"]]["imo"].unique()
    candidates = gold[gold["imo"].isin(same_type_imos)]

    best_imo, best_score, best_reasons = None, 0.0, []
    for _, candidate in candidates.iterrows():
        score, reasons = match_score(row, candidate)
        if score > best_score:
            best_imo, best_score, best_reasons = int(candidate["imo"]), score, reasons

    if best_score >= threshold:
        return best_imo, best_score, best_reasons
    return None, best_score, best_reasons

print("match_score / best_match ready.")

match_score / best_match ready.


In [6]:
keyless = df[~df["imo_valid"]]

# a row that should match: sentinel IMO 0 but named FREEDOM, has dims
case_a = keyless[(keyless["name"].str.upper() == "FREEDOM")
                 & (keyless["length"].notna())].iloc[0]
imo_a, score_a, why_a = best_match(case_a)
print("FREEDOM (imo=0):", case_a[["mmsi", "length", "width", "flag"]].to_dict())
print("  ->", imo_a, score_a, why_a, "| LINK" if imo_a else "| new")

# a row that shouldn't match: 2-char name, nothing to go on
case_b = keyless[keyless["name"] == "XF"].iloc[0]
imo_b, score_b, why_b = best_match(case_b)
print("XF (imo=1000000):", case_b[["mmsi", "length", "flag"]].to_dict())
print("  -> best", score_b, "(< 0.6) | new vessel" if not imo_b else f"| LINK {imo_b}")

FREEDOM (imo=0): {'mmsi': 367416420, 'length': 28.0, 'width': 11.0, 'flag': 'US'}
  -> 9290622 0.75 ['name~1.00', 'length=', 'grossTonnage='] | LINK
XF (imo=1000000): {'mmsi': 212222100, 'length': 399.0, 'flag': 'CY'}
  -> best 0.113 (< 0.6) | new vessel


In [7]:
# step 4: once resolved, filter queries are clearer as sql than pandas - and this
# is the query shape the conversational layer would emit. sqlite here, postgres in practice.
import sqlite3

con = sqlite3.connect(":memory:")
gold_sql = gold.drop(columns=["history", "conflicts"]).copy()
vt = valid_df.groupby("imo")["vessel_type"].agg(
    lambda s: s.dropna().iloc[-1] if not s.dropna().empty else None)
gold_sql["vessel_type"] = gold_sql["imo"].map(vt)
gold_sql.to_sql("vessels", con, index=False, if_exists="replace")

# e.g. search_vessels(type="Dry Bulk", min_length=200)
query = """
    SELECT imo, name, flag, vessel_type, length, grossTonnage
    FROM vessels
    WHERE vessel_type = ? AND length > ?
    ORDER BY length DESC
    LIMIT 5
"""
pd.read_sql_query(query, con, params=("Dry Bulk", 200))

,imo,name,flag,vessel_type,length,grossTonnage
0,9447196,PIGI,CY,Dry Bulk,300.0,107059.0
1,9939383,LADY SOPHIE MARIE,MH,Dry Bulk,299.0,108559.0
2,9355161,MADEIRA,LR,Dry Bulk,292.0,91739.0
3,9566992,GENIUS,CY,Dry Bulk,292.0,91971.0
4,9567037,JEWEL,MH,Dry Bulk,292.0,91971.0


notes:

covers key questions 1-3: same vessel (deterministic imo + fuzzy fallback),
invalid/conflict detection (checksum + flagged immutable conflicts), change
tracking (history per mutable field).

left for the design doc / a real build:
- threshold 0.6 and weights are eyeballed - need a labelled set to tune precision/recall
- blocking on vessel_type only is coarse (real: length-bucket + name-prefix, or ANN index)
- conflict resolution is majority/recency - source-reliability ranking would be better
- reverse conflict (one mmsi under two valid imos) -> flag for review, never auto-merge